# 03 — Обучение модели 2: XLM-RoBERTa

**Модель:** `xlm-roberta-base`

XLM-RoBERTa — многоязычная RoBERTa от Meta, обученная на 100 языках включая русский. Как правило, превосходит mBERT на задачах для конкретных языков.

**Гиперпараметры:**
- batch_size: 16 (модель крупнее ruBERT, нужно больше памяти)
- learning_rate: 2e-5
- epochs: 5 (с early stopping)

In [ ]:
import sys, os

PROJECT_ROOT = "/kaggle/input/sentiment-analysis" if os.path.exists("/kaggle") else os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "../results"
os.makedirs(f'{WORKING_DIR}/models/xlmroberta', exist_ok=True)

In [ ]:
import torch
import warnings
warnings.filterwarnings('ignore')

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Загрузка данных

In [ ]:
from datasets import load_from_disk

DATA_PATH = f'{WORKING_DIR}/processed_data'

if os.path.exists(DATA_PATH):
    dataset = load_from_disk(DATA_PATH)
else:
    from src.data_loader import load_data
    from src.preprocessor import preprocess_batch
    dataset, _ = load_data()
    dataset = dataset.map(lambda b: preprocess_batch(b, clean=True), batched=True, desc='Preprocessing')

for split in dataset:
    print(f'  {split}: {len(dataset[split]):,}')

## Обучение XLM-RoBERTa

In [ ]:
from src.trainer import train_model

MODEL_NAME = 'xlm-roberta-base'
OUTPUT_DIR = f'{WORKING_DIR}/models/xlmroberta'

model, tokenizer, results = train_model(
    model_name=MODEL_NAME,
    dataset=dataset,
    output_dir=OUTPUT_DIR,
    num_labels=3,
    num_epochs=5,
    batch_size=16,          # Меньше batch — модель крупнее
    learning_rate=2e-5,
    max_length=128,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    seed=42,
    early_stopping_patience=2,
)

## Результаты

In [ ]:
from src.evaluation import evaluate_predictions, plot_confusion_matrix
import numpy as np

preds  = np.array(results['predictions'])
labels = np.array(results['true_labels'])
probs  = np.array(results['probabilities'])

metrics = evaluate_predictions(labels, preds, probs, model_name='XLM-RoBERTa')
print('\nМетрики XLM-RoBERTa на тесте:')
for k, v in metrics.items():
    if k != 'model':
        print(f'  {k:<20s}: {v:.4f}')

In [ ]:
plot_confusion_matrix(
    labels, preds,
    model_name='XLM-RoBERTa',
    save_path=f'{WORKING_DIR}/cm_xlmroberta.png',
)
print(f'\nF1-macro = {metrics["f1_macro"]:.4f}')